# Kenya Conversational Engine: Kaggle GPU worker (Sauti V2 TTS + Qwen3.5-4B)

One FastAPI service on Kaggle's **GPU T4 ×2**, exposed through your **ngrok static domain**, implementing the backend's worker contracts:

| Route | Auth | Model | GPU |
|---|---|---|---|
| `POST /synthesize` `{text, voice, format}` → audio | `X-Worker-Key` | Sauti V2 (Chatterbox + LoRA) | `cuda:0` (fp32) |
| `POST /complete` `{messages, ...}` → `{"content": ...}` | `X-Worker-Key` | Qwen3.5-4B, text-only, thinking off, 8K context | `cuda:1` (fp16) |
| `GET /health` → which models are loaded | none | | |

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **on**, and the secrets `HF_TOKEN`, `NGROK_TOKEN`, `NGROK_DOMAIN`, `WORKER_API_KEY` attached (Add-ons → Secrets).

**Faster starts:** run `export_weights.ipynb` once (CPU, no GPU hours), then attach its output here with **Add Input**. The ~13 GB of model weights then load from `/kaggle/input` instead of downloading every session. Without it, they download from Hugging Face.

* **Testing:** *Run All*. The last cell keeps the worker alive while this session is open.
* **Background:** *Save Version → Save & Run All (Commit)*. The committed run keeps serving until Kaggle's session limit. Stop it from the notebook's *Versions/Runs* page.

Secret values are never printed. See `workers/kaggle/README.md` for the full guide.

In [ ]:
# Cell 1: GPU and secrets check (fails fast)
import os
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi is not available")

import torch

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(
        f"Expected 2 GPUs, found {GPU_COUNT}. Set Settings → Accelerator → 'GPU T4 x2', then restart the session."
    )
for _i in range(GPU_COUNT):
    print(f"cuda:{_i} = {torch.cuda.get_device_name(_i)}")

from kaggle_secrets import UserSecretsClient

REQUIRED_SECRETS = ("HF_TOKEN", "NGROK_TOKEN", "NGROK_DOMAIN", "WORKER_API_KEY")
SECRETS = {}
_missing = []
_client = UserSecretsClient()
for _name in REQUIRED_SECRETS:
    try:
        _value = _client.get_secret(_name)
    except Exception:
        _value = None
    if _value and _value.strip():
        SECRETS[_name] = _value.strip()
    else:
        _missing.append(_name)
del _client
if _missing:
    raise RuntimeError(
        "Missing Kaggle secrets: " + ", ".join(_missing)
        + ". Add them under Add-ons → Secrets using exactly these names, and tick 'Attached' for this notebook."
    )

SECRETS["NGROK_DOMAIN"] = SECRETS["NGROK_DOMAIN"].removeprefix("https://").removeprefix("http://").strip("/")
if len(SECRETS["WORKER_API_KEY"]) < 16:
    raise RuntimeError("WORKER_API_KEY must be at least 16 characters (the API refuses shorter keys).")


def redact(text) -> str:
    """Masks every secret value in a string before it is printed (e.g. exception messages)."""
    text = str(text)
    for value in SECRETS.values():
        if value:
            text = text.replace(value, "***")
    return text


os.environ["HF_TOKEN"] = SECRETS["HF_TOKEN"]  # read by huggingface_hub; never printed
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("OK: all secrets present:", ", ".join(REQUIRED_SECRETS))  # names only

In [ ]:
# Cell 2: dependencies (pinned). Takes a few minutes the first time.
# chatterbox-tts 0.1.7 is the same code the chatterbox-finetuning toolkit (which trained Sauti V2) vendors, and it
# targets transformers 5.2.0, the first release with Qwen3.5. --no-deps keeps Kaggle's preinstalled CUDA torch.
%pip install -q --no-deps "chatterbox-tts==0.1.7"
%pip install -q "transformers==5.2.0" "diffusers==0.29.0" "peft>=0.17.1" "s3tokenizer" "resemble-perth==1.0.1" "conformer==0.3.2" "omegaconf" "librosa==0.11.0" "silero-vad" "ngrok==1.7.0" "fastapi" "uvicorn" "soundfile"
# Kaggle preinstalls torchao 0.10, which PEFT rejects ("only versions above 0.16.0 are supported") while applying the
# Sauti LoRA. Nothing here uses torchao; without it installed, PEFT skips that code path.
%pip uninstall -y -q torchao

from importlib.metadata import version as _v

for _pkg in ("torch", "torchaudio", "transformers", "peft", "torchao", "chatterbox-tts", "diffusers", "huggingface-hub", "numpy", "fastapi", "ngrok"):
    try:
        print(f"{_pkg}=={_v(_pkg)}")
    except Exception:
        print(f"{_pkg}: not installed")

In [ ]:
# Cell 3: configuration
PORT = 8000
TTS_DEVICE = "cuda:0"
SLM_DEVICE = "cuda:1"

# Pinned model revisions (immutable versions; bump deliberately).
CHATTERBOX_REPO, CHATTERBOX_REVISION = "ResembleAI/chatterbox", "5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18"
SAUTI_REPO, SAUTI_REVISION = "msingiai/sauti_tts_v2", "09de6d51288c5bf42931c082bcc9ebcbce68ab87"
QWEN_REPO, QWEN_REVISION = "Qwen/Qwen3.5-4B", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

# Sauti V2 = English Chatterbox T3 with its text vocabulary resized to the 2,454-token multilingual grapheme
# tokenizer, plus a LoRA adapter (r=128) that also carries the retrained text_emb/text_head.
SAUTI_TOKENIZER_FILE = "grapheme_mtl_merged_expanded_v1.json"
SAUTI_VOCAB_SIZE = 2454

TTS_MAX_CHARS = 1000        # hard limit per request (keep the API's TTS_MAX_INPUT_CHARACTERS at or below this)
TTS_CHUNK_CHARS = 250       # text is split at sentence boundaries into chunks of at most this many characters
TTS_PAUSE_SECONDS = 0.2     # silence inserted between chunks
TTS_PARAMS = dict(temperature=0.8, repetition_penalty=1.2, min_p=0.05, top_p=1.0, exaggeration=0.5, cfg_weight=0.5)
REFERENCE_VOICE_WAV = None  # optional: path to a clean 5–10 s speaker clip (e.g. from an attached Kaggle dataset)
TRIM_TRAILING_SILENCE = True  # Silero VAD trim of trailing noise/hallucination, as in the fine-tuning toolkit

SLM_DTYPE = "float16"       # T4 has no fast bf16. If replies come back empty or garbled, see the README.
SLM_MAX_CONTEXT_TOKENS = 8192
SLM_DEFAULT_MAX_NEW_TOKENS = 512
SLM_MAX_NEW_TOKENS = 1024
SLM_MAX_INPUT_CHARS = 32000
SLM_MAX_MESSAGES = 100
SLM_SAMPLING = dict(top_p=0.8, top_k=20)  # Qwen3.5 non-thinking recommendation (temperature 0.7)
SLM_DEFAULT_TEMPERATURE = 0.7

In [ ]:
# Cell 3b: model weights: the attached Kaggle input first, Hugging Face as the fallback
# Attach the output of export_weights.ipynb (or a Dataset made from it) via "Add Input" to skip the ~13 GB
# download on every session. Without it, everything downloads from Hugging Face as before.
import glob
import json
from pathlib import Path

WEIGHTS_BUNDLE = "kenya-engine-weights"
KAGGLE_INPUT = "/kaggle/input"
# Keep in sync with export_weights.ipynb.
WEIGHTS = {
    "chatterbox": {
        "repo": CHATTERBOX_REPO,
        "revision": CHATTERBOX_REVISION,
        "allow_patterns": ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "conds.pt", SAUTI_TOKENIZER_FILE],
    },
    "sauti": {"repo": SAUTI_REPO, "revision": SAUTI_REVISION, "allow_patterns": ["new_lang_adapter/*"]},
    "qwen": {"repo": QWEN_REPO, "revision": QWEN_REVISION, "allow_patterns": ["*.json", "*.safetensors", "*.jinja", "*.txt"]},
}
WEIGHT_SOURCES = {}


def required_weight_files(name: str, root: Path) -> list:
    """Files that must exist for a model folder to be usable."""
    if name == "chatterbox":
        return [root / f for f in WEIGHTS["chatterbox"]["allow_patterns"]]
    if name == "sauti":
        return [root / "new_lang_adapter" / f for f in ("adapter_config.json", "adapter_model.safetensors")]
    index = root / "model.safetensors.index.json"
    files = [root / "config.json", root / "tokenizer_config.json", root / "chat_template.jinja", index]
    if index.is_file():
        files += [root / shard for shard in sorted(set(json.loads(index.read_text())["weight_map"].values()))]
    return files


def find_weights_bundle(input_root: str = KAGGLE_INPUT):
    """The attached bundle folder (the one holding our MANIFEST.json), wherever Kaggle mounted it, or None."""
    for manifest in sorted(glob.glob(f"{input_root}/**/MANIFEST.json", recursive=True)):
        try:
            if json.loads(Path(manifest).read_text()).get("bundle") == WEIGHTS_BUNDLE:
                return Path(manifest).parent
        except (OSError, ValueError):
            continue
    return None


WEIGHTS_BUNDLE_DIR = find_weights_bundle()


def weights_dir(name: str) -> Path:
    """Local folder with the pinned weights for `name`: from the attached bundle when it has exactly the pinned
    revision and every file, otherwise downloaded from Hugging Face."""
    spec = WEIGHTS[name]
    if WEIGHTS_BUNDLE_DIR is not None:
        entry = json.loads((WEIGHTS_BUNDLE_DIR / "MANIFEST.json").read_text()).get("models", {}).get(name, {})
        root = WEIGHTS_BUNDLE_DIR / name
        if (entry.get("repo"), entry.get("revision")) != (spec["repo"], spec["revision"]):
            print(f"[weights] {name}: bundle has {entry.get('repo')}@{str(entry.get('revision'))[:7]}, "
                  f"need {spec['repo']}@{spec['revision'][:7]}; downloading instead")
        else:
            missing = [str(p.relative_to(root)) for p in required_weight_files(name, root) if not p.is_file()]
            if not missing:
                WEIGHT_SOURCES[name] = "kaggle-input"
                print(f"[weights] {name}: using attached bundle ({root})")
                return root
            print(f"[weights] {name}: bundle is missing {missing[:3]}; downloading instead")
    from huggingface_hub import snapshot_download

    WEIGHT_SOURCES[name] = "huggingface"
    print(f"[weights] {name}: downloading {spec['repo']}@{spec['revision'][:7]} from Hugging Face")
    return Path(snapshot_download(spec["repo"], revision=spec["revision"], allow_patterns=spec["allow_patterns"]))


print(
    f"Weights bundle attached at {WEIGHTS_BUNDLE_DIR}" if WEIGHTS_BUNDLE_DIR
    else "No weights bundle attached: models will download from Hugging Face (~13 GB). See the README to cache them."
)

In [ ]:
# Cell 4: load Sauti V2 (Chatterbox + LoRA) on cuda:0
import gc
import shutil
import time
import traceback

from safetensors.torch import load_file

MODEL_STATUS = {"tts": False, "slm": False}
MODEL_ERRORS = {}
LOAD_SECONDS = {}
tts_engine = None


def vram(device) -> str:
    free, total = torch.cuda.mem_get_info(device)
    return (
        f"allocated {torch.cuda.memory_allocated(device) / 2**30:.2f} GiB, "
        f"reserved {torch.cuda.memory_reserved(device) / 2**30:.2f} GiB, "
        f"used {(total - free) / 2**30:.2f}/{total / 2**30:.2f} GiB"
    )


def _resize_and_load_t3_weights(new_t3, pretrained_state):
    """Port of chatterbox-finetuning src/model.py: copy the base weights into a T3 with a larger text vocabulary."""
    state = new_t3.state_dict()
    for name, param in pretrained_state.items():
        if name in ("text_emb.weight", "text_head.weight"):
            continue
        if name in state and state[name].shape == param.shape:
            state[name].copy_(param)
        else:
            print(f"  skipped (shape mismatch): {name}")
    for name in ("text_emb.weight", "text_head.weight"):
        old = pretrained_state[name]
        n_old = old.shape[0]
        state[name][:n_old].copy_(old)
        extra = state[name].shape[0] - n_old
        if extra > 0:
            state[name][n_old:].copy_(old.mean(dim=0, keepdim=True).expand(extra, -1))
    new_t3.load_state_dict(state)
    return new_t3


def load_sauti():
    import perth

    if getattr(perth, "PerthImplicitWatermarker", None) is None:
        print("WARNING: the Perth implicit watermarker is unavailable here; generated audio will NOT be watermarked.")
        perth.PerthImplicitWatermarker = perth.DummyWatermarker

    from chatterbox.models.t3 import T3
    from chatterbox.tts import ChatterboxTTS
    from peft import PeftModel

    # 1. Base Chatterbox files, with the multilingual grapheme tokenizer as tokenizer.json (the toolkit's setup.py
    #    does the same rename in standard mode).
    base_files = ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "conds.pt"]
    base_dir = weights_dir("chatterbox")
    ckpt = Path("/tmp/sauti_base")
    ckpt.mkdir(parents=True, exist_ok=True)
    for name in base_files:
        link = ckpt / name
        if link.is_symlink() or link.exists():
            link.unlink()
        link.symlink_to(base_dir / name)
    shutil.copyfile(base_dir / SAUTI_TOKENIZER_FILE, ckpt / "tokenizer.json")

    engine = ChatterboxTTS.from_local(ckpt, device="cpu")

    # 2. Rebuild T3 with the 2,454-token vocabulary and transfer the base weights (toolkit inference.py, steps 1-3).
    pretrained_state = engine.t3.state_dict()
    base_q_proj = pretrained_state["tfmr.layers.0.self_attn.q_proj.weight"].clone()
    hp = engine.t3.hp
    hp.text_tokens_dict_size = SAUTI_VOCAB_SIZE
    t3 = _resize_and_load_t3_weights(T3(hp=hp), pretrained_state)
    del pretrained_state

    # 3. Apply the LoRA adapter plus the retrained embeddings, then merge (as the toolkit's merge_lora.py does).
    adapter_dir = weights_dir("sauti") / "new_lang_adapter"
    t3 = PeftModel.from_pretrained(t3, str(adapter_dir), is_trainable=False).merge_and_unload()

    # 4. Prove the adapter took effect. The public reference Space silently falls back to the base model when its
    #    load call fails; this refuses to serve if that happens.
    trained = load_file(str(adapter_dir / "adapter_model.safetensors"))
    for key, module in (("base_model.model.text_emb.weight", t3.text_emb), ("base_model.model.text_head.weight", t3.text_head)):
        if module.weight.shape[0] != SAUTI_VOCAB_SIZE or not torch.allclose(module.weight.detach().float(), trained[key].float()):
            raise RuntimeError(f"Sauti adapter was not applied ({key.split('.')[-2]} does not match the adapter)")
    if torch.allclose(t3.tfmr.layers[0].self_attn.q_proj.weight.detach().float(), base_q_proj.float()):
        raise RuntimeError("Sauti LoRA weights were not merged (q_proj unchanged)")
    del trained, base_q_proj

    engine.t3 = t3.to(TTS_DEVICE).eval()
    engine.s3gen.to(TTS_DEVICE).eval()
    engine.ve.to(TTS_DEVICE).eval()
    engine.device = TTS_DEVICE
    if REFERENCE_VOICE_WAV:
        engine.prepare_conditionals(REFERENCE_VOICE_WAV, exaggeration=TTS_PARAMS["exaggeration"])
    elif engine.conds is not None:
        engine.conds = engine.conds.to(TTS_DEVICE)
    else:
        raise RuntimeError("No default voice (conds.pt) and REFERENCE_VOICE_WAV is not set")
    return engine


_t0 = time.perf_counter()
try:
    tts_engine = load_sauti()
    LOAD_SECONDS["tts_load"] = time.perf_counter() - _t0
    _t1 = time.perf_counter()
    with torch.inference_mode():
        tts_engine.generate("Habari.", **TTS_PARAMS)  # warm-up
    LOAD_SECONDS["tts_warmup"] = time.perf_counter() - _t1
    MODEL_STATUS["tts"] = True
    print(f"Sauti V2 ready on {TTS_DEVICE}: load {LOAD_SECONDS['tts_load']:.1f}s, warm-up {LOAD_SECONDS['tts_warmup']:.1f}s, sample rate {tts_engine.sr} Hz")
    print(f"  VRAM {TTS_DEVICE}: {vram(TTS_DEVICE)}")
except Exception as exc:
    MODEL_ERRORS["tts"] = redact(f"{type(exc).__name__}: {exc}")
    print("ERROR: Sauti V2 failed to load; /synthesize will return 503.\n ", MODEL_ERRORS["tts"])
    print(redact(traceback.format_exc()))
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 5: load Qwen3.5-4B on cuda:1 (fp16, text-only)
slm_model = None
slm_tokenizer = None


def load_qwen():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    qwen_dir = str(weights_dir("qwen"))  # attached bundle or Hugging Face, pinned to QWEN_REVISION
    tokenizer = AutoTokenizer.from_pretrained(qwen_dir)
    # AutoModelForCausalLM maps Qwen3.5's multimodal checkpoint to the text-only Qwen3_5ForCausalLM: the vision
    # tower is not loaded, which saves VRAM.
    model = AutoModelForCausalLM.from_pretrained(
        qwen_dir,
        dtype=getattr(torch, SLM_DTYPE),
        device_map={"": SLM_DEVICE},
    )
    model.eval()
    return tokenizer, model


_t0 = time.perf_counter()
try:
    slm_tokenizer, slm_model = load_qwen()
    LOAD_SECONDS["slm_load"] = time.perf_counter() - _t0
    print(f"Qwen3.5-4B loaded as {type(slm_model).__name__} ({SLM_DTYPE}) on {SLM_DEVICE} in {LOAD_SECONDS['slm_load']:.1f}s")
    print(f"  VRAM {SLM_DEVICE}: {vram(SLM_DEVICE)}")
    MODEL_STATUS["slm"] = True
except Exception as exc:
    MODEL_ERRORS["slm"] = redact(f"{type(exc).__name__}: {exc}")
    print("ERROR: Qwen3.5-4B failed to load; /complete will return 503.\n ", MODEL_ERRORS["slm"])
    print(redact(traceback.format_exc()))
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 6: inference helpers and the FastAPI app (worker contracts)
import gc
import hmac
import io
import json
import re
import threading
import time
from typing import Literal, Optional

import numpy as np
from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel, Field
from starlette.exceptions import HTTPException as StarletteHTTPException

_WORKER_KEY = SECRETS["WORKER_API_KEY"].encode()
TTS_LOCK = threading.Lock()
SLM_LOCK = threading.Lock()
STATS = {"requests": 0, "errors": 0, "unauthorized": 0}


class WorkerError(Exception):
    """Becomes {"error": {"code", "message"}} with the given HTTP status."""

    def __init__(self, status: int, code: str, message: str):
        super().__init__(message)
        self.status, self.code, self.message = status, code, message


def _is_oom(exc: BaseException) -> bool:
    return type(exc).__name__ == "OutOfMemoryError" or "out of memory" in str(exc).lower()


def _free_gpu_memory():
    gc.collect()
    try:
        import torch

        torch.cuda.empty_cache()
    except Exception:
        pass


def run_model(kind: str, fn):
    """Runs a model call, turning CUDA OOM and generation errors into structured WorkerErrors."""
    try:
        return fn()
    except WorkerError:
        raise
    except Exception as exc:
        if _is_oom(exc):
            _free_gpu_memory()
            print(f"[{kind}] CUDA out of memory")
            raise WorkerError(503, "GPU_OUT_OF_MEMORY", "The GPU ran out of memory. Retry with a shorter input.")
        print(f"[{kind}] generation error: {type(exc).__name__}: {redact(str(exc))[:300]}")
        raise WorkerError(500, "GENERATION_FAILED", "Generation failed.")


# --- TTS ------------------------------------------------------------------------------------------------

def split_for_tts(text: str, max_chars: int = TTS_CHUNK_CHARS) -> list:
    """Sentence-aware chunking: whole sentences packed up to max_chars; over-long sentences split at a comma/space."""
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
    chunks, current = [], ""
    for sentence in sentences:
        while len(sentence) > max_chars:
            cut = max(sentence.rfind(", ", 0, max_chars), sentence.rfind(" ", 0, max_chars))
            if cut <= 0:
                cut = max_chars - 1
            part, sentence = sentence[: cut + 1].strip(), sentence[cut + 1 :].strip()
            if current:
                chunks.append(current)
                current = ""
            chunks.append(part)
        if not sentence:
            continue
        if current and len(current) + 1 + len(sentence) > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        chunks.append(current)
    return chunks


def _tts_generate_chunk(text: str) -> np.ndarray:
    """One Chatterbox generation (model call). Returns float32 mono audio at tts_engine.sr."""
    import torch

    with torch.inference_mode():
        wav = tts_engine.generate(text, **TTS_PARAMS)
    return wav.squeeze().detach().cpu().numpy().astype(np.float32)


_vad = None


def _trim_trailing_noise(audio: np.ndarray, sr: int) -> np.ndarray:
    """Cuts anything after the last detected speech (Chatterbox sometimes adds noise or babble at the end)."""
    global _vad
    if not TRIM_TRAILING_SILENCE or audio.size == 0:
        return audio
    try:
        import torch
        import torchaudio
        from silero_vad import get_speech_timestamps, load_silero_vad

        if _vad is None:
            _vad = load_silero_vad()
        audio16k = torchaudio.functional.resample(torch.from_numpy(audio), sr, 16000)
        stamps = get_speech_timestamps(audio16k, _vad, sampling_rate=16000)
        if not stamps:
            return audio
        end = int(stamps[-1]["end"] * sr / 16000) + int(0.1 * sr)
        return audio[: min(end, audio.size)]
    except Exception as exc:
        print(f"[tts] VAD trim skipped: {type(exc).__name__}")
        return audio


def synthesize(text: str):
    """Text -> (audio, sample_rate, chunk_count). Raises WorkerError on empty/failed generations."""
    sr = tts_engine.sr
    pause = np.zeros(int(TTS_PAUSE_SECONDS * sr), dtype=np.float32)
    pieces = []
    chunks = split_for_tts(text)
    for chunk in chunks:
        audio = _trim_trailing_noise(run_model("tts", lambda: _tts_generate_chunk(chunk)), sr)
        if audio.size < int(0.1 * sr) or not np.isfinite(audio).all():
            raise WorkerError(500, "EMPTY_GENERATION", "The model produced no usable audio.")
        pieces.extend([audio, pause])
    return np.concatenate(pieces[:-1]), sr, len(chunks)


def encode_audio(audio: np.ndarray, sr: int, fmt: str):
    """Returns (bytes, media_type). MP3 via libsndfile when available, otherwise WAV (and says so)."""
    import soundfile as sf

    audio = np.clip(audio, -1.0, 1.0)
    if fmt == "mp3":
        try:
            buf = io.BytesIO()
            sf.write(buf, audio, sr, format="MP3")
            return buf.getvalue(), "audio/mpeg"
        except Exception as exc:
            print(f"[tts] MP3 encoding unavailable ({type(exc).__name__}); returning WAV")
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return buf.getvalue(), "audio/wav"


# --- SLM ------------------------------------------------------------------------------------------------

JSON_INSTRUCTION = "Respond with a single valid JSON object and nothing else."


def strip_thinking(text: str) -> str:
    """Removes <think>...</think> (and a dangling opening/closing tag) defensively: thinking is disabled anyway."""
    out = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    closing = out.lower().rfind("</think>")
    if closing != -1:
        out = out[closing + len("</think>") :]
    opening = out.lower().find("<think>")
    if opening != -1:
        out = out[:opening]
    return out.strip()


def _render_prompt(messages: list) -> str:
    return slm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def _count_prompt_tokens(messages: list) -> int:
    return len(slm_tokenizer(_render_prompt(messages), add_special_tokens=False)["input_ids"])


def _chat_generate(messages: list, max_new_tokens: int, temperature: float) -> str:
    """One Qwen generation (model call). Returns the decoded completion."""
    import torch

    inputs = slm_tokenizer(_render_prompt(messages), return_tensors="pt", add_special_tokens=False).to(SLM_DEVICE)
    sampling = dict(do_sample=True, temperature=temperature, **SLM_SAMPLING) if temperature > 0 else dict(do_sample=False)
    with torch.inference_mode():
        output = slm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=slm_tokenizer.pad_token_id or slm_tokenizer.eos_token_id,
            **sampling,
        )
    return slm_tokenizer.decode(output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


def _extract_json(text: str) -> str:
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            json.loads(match.group(0))
            return match.group(0)
        except ValueError:
            pass
    return text  # the API layer reports invalid JSON as SLM_COMPLETION_FAILED


def complete(messages: list, temperature: Optional[float], max_tokens: Optional[int], response_format: Optional[str]) -> str:
    if response_format == "json_object":
        messages = [{"role": "system", "content": JSON_INSTRUCTION}, *messages]
    max_new = min(max_tokens or SLM_DEFAULT_MAX_NEW_TOKENS, SLM_MAX_NEW_TOKENS)
    prompt_tokens = run_model("slm", lambda: _count_prompt_tokens(messages))
    if prompt_tokens + max_new > SLM_MAX_CONTEXT_TOKENS:
        raise WorkerError(
            413,
            "CONTEXT_TOO_LONG",
            f"The conversation is {prompt_tokens} tokens; with {max_new} new tokens it exceeds the {SLM_MAX_CONTEXT_TOKENS}-token context.",
        )
    temperature = SLM_DEFAULT_TEMPERATURE if temperature is None else temperature
    content = strip_thinking(run_model("slm", lambda: _chat_generate(messages, max_new, temperature)))
    if not content:
        raise WorkerError(500, "EMPTY_GENERATION", "The model produced an empty reply.")
    return _extract_json(content) if response_format == "json_object" else content


# --- HTTP -------------------------------------------------------------------------------------------------

class SynthesizeRequest(BaseModel):
    text: str
    voice: str = "default"
    format: Literal["mp3", "wav"] = "wav"


class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class CompleteRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: Optional[float] = Field(default=None, ge=0, le=2)
    max_tokens: Optional[int] = Field(default=None, ge=1)
    response_format: Optional[Literal["text", "json_object"]] = None


app = FastAPI(title="kenya-engine-kaggle-worker", docs_url=None, redoc_url=None, openapi_url=None)


def _error(status: int, code: str, message: str) -> JSONResponse:
    return JSONResponse(status_code=status, content={"error": {"code": code, "message": message}})


@app.middleware("http")
async def auth_and_log(request: Request, call_next):
    """X-Worker-Key on everything except /health (constant-time compare), then one log line per request.
    Logs method, path, status and latency only: never bodies, text or headers."""
    started = time.perf_counter()
    STATS["requests"] += 1
    if request.url.path != "/health":
        supplied = request.headers.get("x-worker-key", "").encode()
        if not hmac.compare_digest(supplied, _WORKER_KEY):
            STATS["unauthorized"] += 1
            response = _error(401, "UNAUTHORIZED", "Missing or invalid X-Worker-Key.")
            print(f"{request.method} {request.url.path} 401 {1000 * (time.perf_counter() - started):.0f}ms", flush=True)
            return response
    response = await call_next(request)
    if response.status_code >= 400:
        STATS["errors"] += 1
    print(f"{request.method} {request.url.path} {response.status_code} {1000 * (time.perf_counter() - started):.0f}ms", flush=True)
    return response


@app.exception_handler(WorkerError)
async def _worker_error(_request, exc: WorkerError):
    return _error(exc.status, exc.code, exc.message)


@app.exception_handler(RequestValidationError)
async def _validation_error(_request, exc: RequestValidationError):
    fields = sorted({".".join(str(p) for p in err.get("loc", ())[1:]) or "body" for err in exc.errors()})
    return _error(400, "VALIDATION_ERROR", "Invalid request: " + ", ".join(fields))


@app.exception_handler(StarletteHTTPException)
async def _http_error(_request, exc: StarletteHTTPException):
    codes = {404: "NOT_FOUND", 405: "METHOD_NOT_ALLOWED"}
    return _error(exc.status_code, codes.get(exc.status_code, "HTTP_ERROR"), str(exc.detail))


@app.exception_handler(Exception)
async def _unexpected_error(_request, exc: Exception):
    print(f"unexpected error: {type(exc).__name__}: {redact(str(exc))[:300]}")
    return _error(500, "INTERNAL_ERROR", "Unexpected worker error.")


@app.get("/health")
def health():
    loaded = dict(MODEL_STATUS)
    status = "ok" if all(loaded.values()) else ("degraded" if any(loaded.values()) else "unavailable")
    return {"status": status, "models": loaded}


@app.post("/synthesize")
def synthesize_route(req: SynthesizeRequest):
    if not MODEL_STATUS["tts"]:
        raise WorkerError(503, "MODEL_NOT_LOADED", "The text-to-speech model is not loaded.")
    text = req.text.strip()
    if not text:
        raise WorkerError(400, "EMPTY_INPUT", "text must not be empty.")
    if len(text) > TTS_MAX_CHARS:
        raise WorkerError(413, "INPUT_TOO_LONG", f"text must be at most {TTS_MAX_CHARS} characters.")
    if req.voice != "default":
        raise WorkerError(400, "UNSUPPORTED_VOICE", "Only the 'default' voice is available.")
    started = time.perf_counter()
    with TTS_LOCK:
        audio, sr, chunks = synthesize(text)
    body, media_type = encode_audio(audio, sr, req.format)
    return Response(
        content=body,
        media_type=media_type,
        headers={
            "X-Audio-Duration-Seconds": f"{audio.size / sr:.3f}",
            "X-Generation-Seconds": f"{time.perf_counter() - started:.3f}",
            "X-Chunks": str(chunks),
        },
    )


@app.post("/complete")
def complete_route(req: CompleteRequest):
    if not MODEL_STATUS["slm"]:
        raise WorkerError(503, "MODEL_NOT_LOADED", "The language model is not loaded.")
    if not req.messages:
        raise WorkerError(400, "EMPTY_INPUT", "messages must not be empty.")
    if len(req.messages) > SLM_MAX_MESSAGES:
        raise WorkerError(413, "INPUT_TOO_LONG", f"At most {SLM_MAX_MESSAGES} messages are allowed.")
    if any(not m.content.strip() for m in req.messages):
        raise WorkerError(400, "EMPTY_INPUT", "Message content must not be empty.")
    if sum(len(m.content) for m in req.messages) > SLM_MAX_INPUT_CHARS:
        raise WorkerError(413, "INPUT_TOO_LONG", f"The conversation exceeds {SLM_MAX_INPUT_CHARS} characters.")
    messages = [m.model_dump() for m in req.messages]
    with SLM_LOCK:
        content = complete(messages, req.temperature, req.max_tokens, req.response_format)
    return {"content": content}


print("App ready. Routes: GET /health, POST /synthesize, POST /complete")

In [ ]:
# Cell 7: start the server and open the ngrok tunnel on your static domain
import ngrok
import requests
import uvicorn

# Re-running this cell in the same session: stop the previous server and tunnel first.
if globals().get("_uvicorn_server") is not None:
    _uvicorn_server.should_exit = True
    _uvicorn_thread.join(timeout=10)
try:
    ngrok.kill()
except Exception:
    pass

_uvicorn_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning", access_log=False))
_uvicorn_thread = threading.Thread(target=_uvicorn_server.run, name="uvicorn", daemon=True)
_uvicorn_thread.start()
_deadline = time.time() + 30
while not _uvicorn_server.started:
    if time.time() > _deadline or not _uvicorn_thread.is_alive():
        raise RuntimeError("uvicorn did not start")
    time.sleep(0.2)
LOCAL_URL = f"http://127.0.0.1:{PORT}"
print("Server listening on", LOCAL_URL)

try:
    _listener = ngrok.forward(f"127.0.0.1:{PORT}", authtoken=SECRETS["NGROK_TOKEN"], domain=SECRETS["NGROK_DOMAIN"])
except Exception as exc:
    raise RuntimeError(
        "ngrok failed to start: " + redact(exc) + "\nCheck NGROK_TOKEN/NGROK_DOMAIN, and that no other session "
        "(e.g. a committed run) is already serving this domain."
    ) from None

PUBLIC_URL = f"https://{SECRETS['NGROK_DOMAIN']}"
_public = requests.get(f"{PUBLIC_URL}/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=30)
print("Tunnel open on https://<NGROK_DOMAIN>, public /health:", _public.status_code, _public.json())

In [ ]:
# Cell 8: smoke test: English, Kiswahili and code-switched speech, plus one chat turn
SMOKE_TTS = {
    "english": "Good morning. Your order has been dispatched and will arrive in Nairobi tomorrow.",
    "kiswahili": "Habari za asubuhi. Oda yako imetumwa na itafika Nairobi kesho.",
    "code_switched": "Nimefika office, I will call you nikimaliza meeting.",
}
AUTH = {"X-Worker-Key": SECRETS["WORKER_API_KEY"]}
os.makedirs("/kaggle/working", exist_ok=True)


def _timed_post(path, payload, device):
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    res = requests.post(f"{LOCAL_URL}{path}", json=payload, headers=AUTH, timeout=600)
    return res, time.perf_counter() - started, torch.cuda.max_memory_allocated(device) / 2**30


for name, text in SMOKE_TTS.items():
    try:
        res, seconds, peak = _timed_post("/synthesize", {"text": text, "voice": "default", "format": "wav"}, TTS_DEVICE)
        if res.status_code != 200:
            print(f"FAIL tts/{name}: HTTP {res.status_code} {res.text[:200]}")
            continue
        path = f"/kaggle/working/smoke_{name}.wav"
        with open(path, "wb") as fh:
            fh.write(res.content)
        audio_s = float(res.headers["X-Audio-Duration-Seconds"])
        print(f"PASS tts/{name}: {seconds:.1f}s for {audio_s:.1f}s audio (RTF {seconds / max(audio_s, 1e-6):.2f}), peak VRAM {peak:.2f} GiB -> {path}")
    except Exception as exc:
        print(f"FAIL tts/{name}: {type(exc).__name__}: {redact(exc)[:200]}")
print(f"  VRAM {TTS_DEVICE}: {vram(TTS_DEVICE)}")

try:
    res, seconds, peak = _timed_post(
        "/complete",
        {"messages": [
            {"role": "system", "content": "Wewe ni msaidizi wa huduma kwa wateja nchini Kenya. Jibu kwa ufupi."},
            {"role": "user", "content": "Nimefika office, nataka kujua status ya order yangu."},
        ], "max_tokens": 128},
        SLM_DEVICE,
    )
    if res.status_code == 200:
        print(f"PASS slm: {seconds:.1f}s, peak VRAM {peak:.2f} GiB\n  reply: {res.json()['content'][:300]}")
    else:
        print(f"FAIL slm: HTTP {res.status_code} {res.text[:200]}")
except Exception as exc:
    print(f"FAIL slm: {type(exc).__name__}: {redact(exc)[:200]}")
print(f"  VRAM {SLM_DEVICE}: {vram(SLM_DEVICE)}")

_unauth = requests.post(f"{LOCAL_URL}/complete", json={"messages": [{"role": "user", "content": "hi"}]}, timeout=30)
print(("PASS" if _unauth.status_code == 401 else "FAIL") + f" auth: request without X-Worker-Key -> {_unauth.status_code}")
print("Load/warm-up seconds:", {k: round(v, 1) for k, v in LOAD_SECONDS.items()})
print("Weights loaded from:", WEIGHT_SOURCES)
if MODEL_ERRORS:
    print("Model load errors:", MODEL_ERRORS)

In [ ]:
# Cell 9: keep the worker alive (required for a committed/background run; interrupt to stop)
print("Worker running. Verify from anywhere: https://<NGROK_DOMAIN>/health")
_last = 0.0
while True:
    if not _uvicorn_thread.is_alive():
        raise RuntimeError("The API server thread stopped; re-run cells 7-9.")
    if time.time() - _last >= 600:
        _last = time.time()
        print(
            time.strftime("%H:%M:%S"),
            f"requests={STATS['requests']} errors={STATS['errors']} unauthorized={STATS['unauthorized']}",
            f"| {TTS_DEVICE}: {vram(TTS_DEVICE)} | {SLM_DEVICE}: {vram(SLM_DEVICE)}",
            flush=True,
        )
    time.sleep(30)